# ANN & Deep Learning — CNN from Scratch in PyTorch

**Datasets:** MNIST → CIFAR-10  
**Architecture:** 2 convolutional layers + Batch Normalization + Max Pooling + Dropout + Fully Connected classifier

## Objectives
- Build a CNN from scratch in PyTorch.
- Train it on MNIST.
- Adapt the CNN for CIFAR-10.
- Record training/validation accuracy and loss.
- Visualize learned first-layer filters.
- Visualize feature maps after the first convolutional layer.


In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES AND SET DEVICE
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import numpy as np

print("PyTorch version:", torch.__version__)

# Use GPU in Colab if available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


## Part A — MNIST

MNIST images are grayscale, so each image has **1 input channel** and a size of **28×28**.

The CNN pipeline is:

`1×28×28 → Conv → BatchNorm → ReLU → MaxPool → Conv → BatchNorm → ReLU → MaxPool → Flatten → FC → Dropout → FC(10)`


In [ ]:
# ============================================================
# 2. LOAD AND PREPROCESS MNIST
# ============================================================

mnist_transform = transforms.Compose([
    transforms.ToTensor(),

    # Normalize MNIST pixel values.
    transforms.Normalize((0.1307,), (0.3081,))
])

mnist_full = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=mnist_transform
)

mnist_test = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=mnist_transform
)

# 50,000 training + 10,000 validation
mnist_train, mnist_val = random_split(
    mnist_full,
    [50000, 10000],
    generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 128

mnist_train_loader = DataLoader(
    mnist_train, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

mnist_val_loader = DataLoader(
    mnist_val, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

mnist_test_loader = DataLoader(
    mnist_test, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

print("Training:", len(mnist_train))
print("Validation:", len(mnist_val))
print("Test:", len(mnist_test))


In [ ]:
# ============================================================
# 3. VISUALIZE MNIST SAMPLES
# ============================================================

images, labels = next(iter(mnist_train_loader))

plt.figure(figsize=(10, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(f"Label: {labels[i].item()}")
    plt.axis("off")

plt.suptitle("MNIST Samples")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 4. CNN ARCHITECTURE
# ============================================================
#
# MNIST dimensions:
#
# Input:       1 x 28 x 28
# Conv1:      32 x 28 x 28
# MaxPool:    32 x 14 x 14
# Conv2:      64 x 14 x 14
# MaxPool:    64 x  7 x  7
# Flatten:        64*7*7 = 3136
# FC1:              128
# Dropout:           128
# FC2:                10
#
# ============================================================

class CNN(nn.Module):

    def __init__(self, in_channels, num_classes, image_size):
        super().__init__()

        # -----------------------------
        # CONVOLUTIONAL BLOCK 1
        # -----------------------------
        self.conv1 = nn.Conv2d(
            in_channels,
            32,
            kernel_size=3,
            padding=1
        )

        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)

        # -----------------------------
        # CONVOLUTIONAL BLOCK 2
        # -----------------------------
        self.conv2 = nn.Conv2d(
            32,
            64,
            kernel_size=3,
            padding=1
        )

        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)

        # Calculate flattened size automatically.
        # This allows the same class to work for MNIST
        # and CIFAR-10.

        with torch.no_grad():
            dummy = torch.zeros(
                1, in_channels, image_size, image_size
            )

            dummy = self.pool1(
                self.relu1(
                    self.bn1(
                        self.conv1(dummy)
                    )
                )
            )

            dummy = self.pool2(
                self.relu2(
                    self.bn2(
                        self.conv2(dummy)
                    )
                )
            )

            flattened_size = dummy.view(1, -1).shape[1]

        print(
            f"Flattened feature size for {image_size}x{image_size}:",
            flattened_size
        )

        # -----------------------------
        # FULLY CONNECTED CLASSIFIER
        # -----------------------------
        self.fc1 = nn.Linear(
            flattened_size,
            128
        )

        self.relu3 = nn.ReLU()

        # Drop 30% of activations during training.
        self.dropout = nn.Dropout(p=0.3)

        self.fc2 = nn.Linear(
            128,
            num_classes
        )

    def forward(self, x):

        # Convolutional block 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        # Convolutional block 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        # Flatten feature maps
        x = torch.flatten(x, start_dim=1)

        # Fully connected classifier
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.dropout(x)

        # 10 class logits
        x = self.fc2(x)

        return x


# Create MNIST CNN
mnist_model = CNN(
    in_channels=1,
    num_classes=10,
    image_size=28
).to(device)

print("\nMNIST MODEL")
print(mnist_model)


In [ ]:
# ============================================================
# 5. CHECK MNIST CNN SHAPES
# ============================================================

sample_images, _ = next(iter(mnist_train_loader))
sample_images = sample_images[:4].to(device)

with torch.no_grad():
    x = mnist_model.conv1(sample_images)
    print("After Conv1:   ", x.shape)

    x = mnist_model.bn1(x)
    x = mnist_model.relu1(x)
    x = mnist_model.pool1(x)
    print("After Pool1:   ", x.shape)

    x = mnist_model.conv2(x)
    print("After Conv2:   ", x.shape)

    x = mnist_model.bn2(x)
    x = mnist_model.relu2(x)
    x = mnist_model.pool2(x)
    print("After Pool2:   ", x.shape)

    x = torch.flatten(x, start_dim=1)
    print("After Flatten: ", x.shape)


In [ ]:
# ============================================================
# 6. TRAINING / VALIDATION FUNCTIONS
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward propagation
        outputs = model(images)

        # Classification loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    return running_loss / total, 100 * correct / total


def evaluate(model, loader, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    # No gradients are needed for validation/testing.
    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return running_loss / total, 100 * correct / total


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs
):

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(epochs):

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer
        )

        val_loss, val_acc = evaluate(
            model, val_loader, criterion
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)

        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}%"
        )

    return history


In [ ]:
# ============================================================
# 7. TRAIN MNIST CNN
# ============================================================

criterion = nn.CrossEntropyLoss()

mnist_optimizer = optim.Adam(
    mnist_model.parameters(),
    lr=0.001
)

# 5 epochs is suitable for a classroom demonstration.
MNIST_EPOCHS = 5

print("=" * 70)
print("TRAINING CNN ON MNIST")
print("=" * 70)

mnist_history = train_model(
    mnist_model,
    mnist_train_loader,
    mnist_val_loader,
    criterion,
    mnist_optimizer,
    MNIST_EPOCHS
)

mnist_test_loss, mnist_test_acc = evaluate(
    mnist_model,
    mnist_test_loader,
    criterion
)

print("\nMNIST Test Loss:", round(mnist_test_loss, 4))
print("MNIST Test Accuracy:", round(mnist_test_acc, 2), "%")


In [ ]:
# ============================================================
# 8. MNIST LEARNING CURVES
# ============================================================

epochs_range = range(1, MNIST_EPOCHS + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, mnist_history["train_acc"],
         marker="o", label="Training Accuracy")
plt.plot(epochs_range, mnist_history["val_acc"],
         marker="o", label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("MNIST CNN - Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, mnist_history["train_loss"],
         marker="o", label="Training Loss")
plt.plot(epochs_range, mnist_history["val_loss"],
         marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("MNIST CNN - Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# ============================================================
# 9. VISUALIZE MNIST CONV1 FILTERS
# ============================================================
#
# Conv1 weights have shape:
#     [32 filters, 1 input channel, 3, 3]
#
# These are learned low-level filters.
# ============================================================

filters = mnist_model.conv1.weight.detach().cpu()

print("Conv1 filter tensor shape:", filters.shape)

plt.figure(figsize=(10, 8))

for i in range(32):

    plt.subplot(4, 8, i + 1)
    plt.imshow(filters[i, 0], cmap="gray")
    plt.title(f"F{i+1}")
    plt.axis("off")

plt.suptitle("Learned First-Layer Filters — MNIST")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 10. VISUALIZE MNIST FEATURE MAPS AFTER CONV1
# ============================================================

# Select one MNIST test image.
sample_image, sample_label = mnist_test[0]

# Add batch dimension:
# [1, 28, 28] -> [1, 1, 28, 28]
sample_input = sample_image.unsqueeze(0).to(device)

mnist_model.eval()

with torch.no_grad():

    feature_maps = mnist_model.conv1(sample_input)
    feature_maps = mnist_model.bn1(feature_maps)
    feature_maps = mnist_model.relu1(feature_maps)

# Remove batch dimension.
feature_maps = feature_maps.cpu().squeeze(0)

print("Actual digit:", sample_label)
print("Feature map shape:", feature_maps.shape)

plt.figure(figsize=(12, 8))

for i in range(16):

    plt.subplot(4, 4, i + 1)
    plt.imshow(feature_maps[i], cmap="viridis")
    plt.title(f"Feature Map {i+1}")
    plt.axis("off")

plt.suptitle(
    f"MNIST First Conv-Layer Feature Maps | Actual: {sample_label}"
)

plt.tight_layout()
plt.show()


# Part B — CIFAR-10

CIFAR-10 contains **RGB 32×32 images** and 10 classes.

Classes:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`

The CNN can be reused, but the input and image dimensions change:

- MNIST: `1 × 28 × 28`
- CIFAR-10: `3 × 32 × 32`

The model's flattened size is calculated automatically, so the same CNN class can adapt to both datasets.


In [ ]:
# ============================================================
# 11. LOAD AND PREPROCESS CIFAR-10
# ============================================================

cifar_transform = transforms.Compose([
    transforms.ToTensor(),

    # Standard CIFAR-10 normalization
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

cifar_full = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=cifar_transform
)

cifar_test = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=cifar_transform
)

# 45,000 training + 5,000 validation
cifar_train, cifar_val = random_split(
    cifar_full,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42)
)

cifar_train_loader = DataLoader(
    cifar_train, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

cifar_val_loader = DataLoader(
    cifar_val, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

cifar_test_loader = DataLoader(
    cifar_test, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

print("Training:", len(cifar_train))
print("Validation:", len(cifar_val))
print("Test:", len(cifar_test))


In [ ]:
# ============================================================
# 12. VISUALIZE CIFAR-10 SAMPLES
# ============================================================

cifar_classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

images, labels = next(iter(cifar_train_loader))

mean = np.array([0.4914, 0.4822, 0.4465])
std = np.array([0.2470, 0.2435, 0.2616])

plt.figure(figsize=(10, 5))

for i in range(10):

    plt.subplot(2, 5, i + 1)

    # Tensor: [C,H,W] -> NumPy image: [H,W,C]
    image = images[i].permute(1, 2, 0).numpy()

    # Undo normalization for display.
    image = image * std + mean
    image = np.clip(image, 0, 1)

    plt.imshow(image)
    plt.title(cifar_classes[labels[i].item()])
    plt.axis("off")

plt.suptitle("CIFAR-10 Samples")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 13. CREATE CIFAR-10 CNN
# ============================================================

# CIFAR-10:
#   3 input channels
#   32 x 32 image
#   10 output classes

cifar_model = CNN(
    in_channels=3,
    num_classes=10,
    image_size=32
).to(device)

print("\nCIFAR-10 MODEL")
print(cifar_model)


In [ ]:
# ============================================================
# 14. CHECK CIFAR-10 SHAPES
# ============================================================

sample_images, _ = next(iter(cifar_train_loader))
sample_images = sample_images[:4].to(device)

with torch.no_grad():

    x = cifar_model.conv1(sample_images)
    print("After Conv1:   ", x.shape)

    x = cifar_model.bn1(x)
    x = cifar_model.relu1(x)
    x = cifar_model.pool1(x)
    print("After Pool1:   ", x.shape)

    x = cifar_model.conv2(x)
    print("After Conv2:   ", x.shape)

    x = cifar_model.bn2(x)
    x = cifar_model.relu2(x)
    x = cifar_model.pool2(x)
    print("After Pool2:   ", x.shape)

    x = torch.flatten(x, start_dim=1)
    print("After Flatten: ", x.shape)


In [ ]:
# ============================================================
# 15. TRAIN CIFAR-10 CNN
# ============================================================

cifar_optimizer = optim.Adam(
    cifar_model.parameters(),
    lr=0.001
)

CIFAR_EPOCHS = 5

print("=" * 70)
print("TRAINING CNN ON CIFAR-10")
print("=" * 70)

cifar_history = train_model(
    cifar_model,
    cifar_train_loader,
    cifar_val_loader,
    criterion,
    cifar_optimizer,
    CIFAR_EPOCHS
)

cifar_test_loss, cifar_test_acc = evaluate(
    cifar_model,
    cifar_test_loader,
    criterion
)

print("\nCIFAR-10 Test Loss:", round(cifar_test_loss, 4))
print("CIFAR-10 Test Accuracy:", round(cifar_test_acc, 2), "%")


In [ ]:
# ============================================================
# 16. CIFAR-10 LEARNING CURVES
# ============================================================

epochs_range = range(1, CIFAR_EPOCHS + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, cifar_history["train_acc"],
         marker="o", label="Training Accuracy")
plt.plot(epochs_range, cifar_history["val_acc"],
         marker="o", label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("CIFAR-10 CNN - Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, cifar_history["train_loss"],
         marker="o", label="Training Loss")
plt.plot(epochs_range, cifar_history["val_loss"],
         marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CIFAR-10 CNN - Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# ============================================================
# 17. VISUALIZE CIFAR-10 CONV1 FILTERS
# ============================================================
#
# CIFAR-10 Conv1 filters have shape:
#     [32, 3, 3, 3]
#
# Each filter contains weights for:
#     Red channel
#     Green channel
#     Blue channel
#
# We display each filter as an RGB image.
# ============================================================

filters = cifar_model.conv1.weight.detach().cpu()

print("Conv1 filter tensor shape:", filters.shape)

plt.figure(figsize=(10, 8))

for i in range(32):

    plt.subplot(4, 8, i + 1)

    filter_image = filters[i].permute(1, 2, 0).numpy()

    # Normalize individual filter for visualization.
    filter_min = filter_image.min()
    filter_max = filter_image.max()

    filter_image = (
        filter_image - filter_min
    ) / (
        filter_max - filter_min + 1e-8
    )

    plt.imshow(filter_image)
    plt.title(f"F{i+1}")
    plt.axis("off")

plt.suptitle("Learned First-Layer Filters — CIFAR-10")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 18. VISUALIZE CIFAR-10 FEATURE MAPS AFTER CONV1
# ============================================================

sample_image, sample_label = cifar_test[0]

# [3,32,32] -> [1,3,32,32]
sample_input = sample_image.unsqueeze(0).to(device)

cifar_model.eval()

with torch.no_grad():

    feature_maps = cifar_model.conv1(sample_input)
    feature_maps = cifar_model.bn1(feature_maps)
    feature_maps = cifar_model.relu1(feature_maps)

feature_maps = feature_maps.cpu().squeeze(0)

print("Actual class:", cifar_classes[sample_label])
print("Feature map shape:", feature_maps.shape)

plt.figure(figsize=(12, 8))

for i in range(16):

    plt.subplot(4, 4, i + 1)
    plt.imshow(feature_maps[i], cmap="viridis")
    plt.title(f"Feature Map {i+1}")
    plt.axis("off")

plt.suptitle(
    "CIFAR-10 First Conv-Layer Feature Maps | "
    f"Actual: {cifar_classes[sample_label]}"
)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 19. FINAL COMPARISON
# ============================================================

print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(f"MNIST Test Accuracy:    {mnist_test_acc:.2f}%")
print(f"CIFAR-10 Test Accuracy: {cifar_test_acc:.2f}%")

print("\nDataset comparison:")
print("MNIST    -> 1 channel, 28x28, grayscale")
print("CIFAR-10 -> 3 channels, 32x32, RGB")


# Lab Activities

### Activity 1 — Change the Number of Filters
Change the first convolution from 32 filters to 64 filters. Explain how the feature-map dimensions change.

### Activity 2 — Change Dropout
Compare `p=0.1`, `p=0.3`, and `p=0.5`. Record training and validation accuracy.

### Activity 3 — Remove Batch Normalization
Train a version without BatchNorm and compare its learning curves with the original model.

### Activity 4 — Feature-map Analysis
Choose one MNIST image and one CIFAR-10 image. Compare their first-layer feature maps.

### Activity 5 — Explain the Difference
Why does the same small CNN generally find CIFAR-10 more difficult than MNIST?

## Viva Questions
1. What is a CNN?
2. Why are CNNs suitable for images?
3. What is a convolution filter?
4. What does `kernel_size=3` mean?
5. What is padding?
6. What is Max Pooling?
7. Why does pooling reduce spatial dimensions?
8. What is Batch Normalization?
9. What is Dropout?
10. What is a feature map?
11. What is the difference between a filter and a feature map?
12. Why does MNIST have one input channel?
13. Why does CIFAR-10 have three input channels?
14. Why are there 10 output neurons for both datasets?
15. Why is CIFAR-10 more difficult than MNIST?
16. Why do we use `model.train()`?
17. Why do we use `model.eval()`?
18. Why do we use `torch.no_grad()` during evaluation?
19. What does `loss.backward()` do?
20. What happens to the image dimensions after each MaxPool layer?

## Core CNN Pipeline

**Image → Convolution → Batch Normalization → ReLU → Max Pooling → Convolution → Batch Normalization → ReLU → Max Pooling → Flatten → Fully Connected → Dropout → Output**
